## BB Match Scraper
This notebook extracts the full dynamic HTML content from the specified basketball matches page using Selenium.

# libraries and functions


In [ ]:
# 1. Install and import required libraries
# %pip install selenium webdriver-manager beautifulsoup4 --quiet
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
from bs4 import BeautifulSoup
from IPython.display import display, HTML
import pandas as pd
import requests



In [ ]:
# 🕐 ENHANCED SCRAPER: Wait longer for dynamic content like player names
def enhanced_dynamic_scraper(url, wait_time=15):

    driver = None
    try:
        print("🚀 Enhanced scraper for dynamic content...")
        
        # Chrome options optimized for dynamic content
        chrome_options = Options()
        # run non-headless to allow full rendering/clicks (headless may miss some dynamic UI)
        # chrome_options.add_argument("--headless")  # commented out on purpose
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option("useAutomationExtension", False)
        chrome_options.add_argument("--disable-blink-features=AutomationControlled")
        chrome_options.add_argument("--remote-debugging-port=9222")
        chrome_options.add_argument("--enable-logging")
        chrome_options.add_argument("--log-level=0")
        prefs = {
            "profile.default_content_setting_values.notifications": 2,  # block notifications
            "profile.default_content_setting_values.images": 2,         # optionally disable images to speed up
        }
        chrome_options.add_experimental_option("prefs", prefs)
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
        
        # Create driver
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        driver.set_page_load_timeout(60)  # Longer timeout
        
        print(f"🌐 Loading URL: {url}")
        driver.get(url)
        
        # Wait for basic page structure
        print("⏳ Waiting for basic page structure...")
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        
        # Wait for Angular/JavaScript to initialize
        print("⏳ Waiting for Angular/JavaScript to load...")
        time.sleep(5)
        
        # Try to wait for specific content that indicates the page is fully loaded
        try:
            # Wait for tables with ng-binding class (AngularJS content)
            WebDriverWait(driver, wait_time).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, ".ng-binding")) > 0
            )
            print("✅ AngularJS content detected!")
        except:
            print("⚠️ AngularJS content not detected, but continuing...")
        
        # Try to wait for table content specifically
        try:
            WebDriverWait(driver, wait_time).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, "td.ng-binding")) > 0
            )
            print("✅ Table cells with ng-binding detected!")
        except:
            print("⚠️ Table cells not detected, but continuing...")
        
        # Additional wait for any late-loading content
        print(f"⏳ Additional wait ({wait_time} seconds) for dynamic content...")
        time.sleep(wait_time)
        
        # Try to trigger any lazy loading by scrolling
        print("📜 Scrolling to trigger lazy loading...")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(3)
        
        # Click on different tabs to ensure all content is loaded
        print("🔄 Trying to activate different tabs...")
        try:
            # Try to click on team tabs
            tabs = driver.find_elements(By.CSS_SELECTOR, "a[ng-click*='tabSel']")
            for i, tab in enumerate(tabs[3:4]):  # Try first  tab 3 (the fourth)
                try:
                    tab.click()
                    time.sleep(2)
                    print(f"✅ Clicked tab {i+1}")
                except:
                    print(f"⚠️ Could not click tab {i+1}")
        except:
            print("⚠️ No tabs found")
        
        # Final wait
        time.sleep(5)
        
        # Get the final HTML
        html_content = driver.page_source
        
        print(f"✅ Enhanced scraper retrieved {len(html_content):,} characters")
        return html_content
        
    except Exception as e:
        print(f"❌ Enhanced scraper failed: {e}")
        return None
    
    finally:
        if driver:
            try:
                driver.quit()
            except:
                pass



In [ ]:
# 🎯 UNIFIED EXPORT FUNCTION - Handle both "all" and "2W" spans
def export_matches(payload, span="all"):
    """
    Export volleyball matches to CSV format
    
    Args:
        payload: List of match dictionaries from API
        span: "all" for all matches, "2W" for next 2 weeks only
        
    Returns:
        str: Filename of exported file, or None if failed
    """
    print(f"📊 Starting export with span: {span}")
    
    if not payload or not isinstance(payload, list):
        print("❌ No valid payload data provided")
        return None
    
    try:
        import csv
        import pandas as pd
        from datetime import datetime, timedelta
        
        # Calculate date window for filtering
        if span == "2W":
            now = pd.Timestamp.now().normalize()
            end = now + pd.Timedelta(days=7)
            print(f"📅 Date window: {now.date()} → {end.date()}")
            filename = 'matches_next_2weeks.csv'
        else:
            now = None
            end = None
            filename = 'matches_all.csv'
        
        # Filter and process matches
        export_matches = []
        total_processed = 0
        
        for item in payload:
            total_processed += 1
            
            if not isinstance(item, dict):
                continue
            
            # If filtering for 2 weeks, check date
            if span == "2W" and 'datumString' in item and 'beginTijd' in item:
                try:
                    # Parse the date and time
                    date_str = item['datumString']
                    time_str = item.get('beginTijd', '00:00').replace('.', ':')
                    
                    # Combine and parse datetime
                    dt_str = f"{date_str} {time_str}"
                    match_dt = pd.to_datetime(dt_str, format='%d-%m-%Y %H:%M', dayfirst=True)
                    
                    # Check if in our window
                    if not (now <= match_dt <= end):
                        continue  # Skip this match
                        
                except Exception as e:
                    print(f"⚠️ Could not parse date for match {total_processed}: {e}")
                    continue
            
            # Clean the record for export
            clean_record = {}
            for key, value in item.items():
                # Convert any complex objects to strings
                clean_record[key] = str(value) if value is not None else ""
            
            export_matches.append(clean_record)
        
        # Sort matches by date if possible
        if export_matches and 'datumString' in export_matches[0]:
            try:
                export_matches.sort(key=lambda x: pd.to_datetime(x['datumString'], format='%d-%m-%Y', dayfirst=True))
            except:
                print("⚠️ Could not sort by date, keeping original order")
        
        # Export to CSV
        if export_matches:
            with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
                fieldnames = export_matches[0].keys()
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                
                for match in export_matches:
                    writer.writerow(match)
            
            print(f"✅ SUCCESS! Exported {len(export_matches)} matches to '{filename}'")
            print(f"📊 Processed {total_processed} total records")
            
            # Show preview of exported data
            if len(export_matches) > 0:
                print(f"\n👀 Preview of exported matches:")
                preview_cols = ['datumString', 'beginTijd', 'tTNaam', 'tUNaam', 'pouleNaam']
                available_cols = [col for col in preview_cols if col in export_matches[0]]
                
                for i, match in enumerate(export_matches[:5], 1):
                    preview_data = [match.get(col, 'N/A') for col in available_cols]
                    print(f"  {i}. {' | '.join(preview_data)}")
                
                if len(export_matches) > 5:
                    print(f"     ... and {len(export_matches) - 5} more matches")
            
            return filename
        else:
            print(f"❌ No matches found for span '{span}'")
            return None
            
    except Exception as e:
        print(f"❌ Export failed: {e}")
        return None


In [ ]:
# PARSE OFFICIALS
def parse_officials_from_html(enhanced_html):
    """
    Parse the dynamic tab content HTML and return officials tables.

    Args:
        enhanced_html (str): HTML string containing the tab-content.

    Returns:
        tuple: (officials_indexed, Officials_transposed, tab_sections_Officials, split_Officials)
            - officials_indexed: DataFrame indexed by REFFUNCTIE
            - Officials_transposed: transposed DataFrame (officials_indexed.T reset)
            - tab_sections_Officials: intermediate DataFrame with cells_text, NAAM, REFFUNCTIE
            - split_Officials: DataFrame with NAAM and REFFUNCTIE produced by splitting cells_text
    """

    html = enhanced_html or globals().get('enhanced_html') or globals().get('enhanced_html_clean') or None
    if not html:
        raise ValueError("No HTML provided and no `enhanced_html` or `enhanced_html_clean` found in globals.")

    soup = BeautifulSoup(html, "html.parser")
    tab_container = soup.find("div", class_="tab-content")
    if tab_container is None:
        raise ValueError("No div.tab-content found in the HTML.")

    rows = []
    for sec in tab_container.find_all(recursive=False):
        sec_id = sec.get("id") or ""
        ng_show = sec.get("ng-show") or ""
        header = ""
        h3 = sec.find("h3", class_="ng-binding")
        if h3:
            header = h3.get_text(strip=True)
        officials = ""
        off_div = sec.find(id="off")
        if off_div:
            officials = off_div.get_text(" ", strip=True)
        else:
            h4 = sec.find("h4", class_="ng-binding")
            if h4:
                officials = h4.get_text(" ", strip=True)
        iframe = sec.find("iframe")
        iframe_src = iframe.get("src") if iframe else (iframe.get("ng-src") if iframe else "")

        tables = sec.find_all("table")
        if tables:
            for ti, table in enumerate(tables, start=1):
                ths = [th.get_text(" ", strip=True) for th in table.find_all("th")]
                tbody = table.find("tbody")
                trs = tbody.find_all("tr") if tbody else table.find_all("tr")
                for ri, tr in enumerate(trs, start=1):
                    cells = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
                    cells_join = " | ".join([c for c in cells if c])
                    rows.append({
                        "section_id": sec_id,
                        "ng_show": ng_show,
                        "header": header,
                        "officials": officials,
                        "iframe_src": iframe_src,
                        "table_index": ti,
                        "table_headers": " | ".join(ths),
                        "row_index": ri,
                        "cells": cells,
                        "cells_text": cells_join
                    })
        else:
            text = sec.get_text(" ", strip=True)
            rows.append({
                "section_id": sec_id,
                "ng_show": ng_show,
                "header": header,
                "officials": officials,
                "iframe_src": iframe_src,
                "table_index": None,
                "table_headers": None,
                "row_index": None,
                "cells": [],
                "cells_text": text
            })

    tab_sections_Officials = pd.DataFrame(rows)

    # Filter to only the "Official | Functie" table and keep just headers + cells text
    _wanted = "Official | Functie"
    if 'table_headers' in tab_sections_Officials.columns:
        tab_sections_Officials = tab_sections_Officials.loc[
            tab_sections_Officials['table_headers'] == _wanted,
            ['section_id','header','officials','cells_text']
        ].reset_index(drop=True)
    else:
        tab_sections_Officials = pd.DataFrame(columns=['section_id','header','officials','cells_text'])

    def _split_name_ref(text):
        if pd.isna(text) or str(text).strip() == "":
            return pd.Series({"NAAM": pd.NA, "REFFUNCTIE": pd.NA})
        s = str(text).strip()
        parts = [p.strip() for p in s.split("|", 1)]
        if len(parts) == 2:
            naam = parts[0] if parts[0] != "" else pd.NA
            ref = parts[1] if parts[1] != "" else pd.NA
            return pd.Series({"NAAM": naam, "REFFUNCTIE": ref})
        else:
            return pd.Series({"NAAM": pd.NA, "REFFUNCTIE": parts[0] if parts[0] != "" else pd.NA})

    if 'cells_text' in tab_sections_Officials.columns:
        split_Officials = tab_sections_Officials['cells_text'].apply(_split_name_ref)
        tab_sections_Officials = pd.concat([tab_sections_Officials.reset_index(drop=True), split_Officials.reset_index(drop=True)], axis=1)
    else:
        split_Officials = pd.DataFrame(columns=['NAAM', 'REFFUNCTIE'])

    officials = tab_sections_Officials.loc[:, [c for c in ['section_id','header','officials','NAAM','REFFUNCTIE'] if c in tab_sections_Officials.columns]]

    officials_indexed = officials.set_index('REFFUNCTIE')
    Officials_transposed = officials_indexed.T.reset_index(drop=True)

    return officials_indexed, Officials_transposed, tab_sections_Officials, split_Officials




In [ ]:

def get_officials_promoted_df(WEDSTRIJD, wait_time=5):
    """
    Scrape officials for a given match ID (WEDSTRIJD), parse, and return a DataFrame
    with the first row promoted to column names and the second row as the first data row.
    """
    url = f"https://vblweb.wisseq.eu/Home/MatchDetail?wedguid={WEDSTRIJD}"
    enhanced_html = enhanced_dynamic_scraper(url, wait_time=wait_time)
    officials = parse_officials_from_html(enhanced_html)
    # Normalize parse output to a simple DataFrame with only REFFUNCTIE and NAAM
    if isinstance(officials, (list, tuple)) and len(officials) >= 4:
        candidate = officials[3]
    else:
        candidate = officials
    if not isinstance(candidate, pd.DataFrame):
        candidate = globals().get('split_Officials', pd.DataFrame())
    # map columns case-insensitively
    col_map = {c.lower(): c for c in candidate.columns}
    reff_col = None
    name_col = None
    for k, orig in col_map.items():
        if 'reffunct' in k or 'ref' in k or 'functie' in k or 'funct' in k:
            reff_col = orig
        if 'naam' in k or 'name' in k:
            name_col = orig
    # fallback attempts
    if reff_col is None:
        for k, orig in col_map.items():
            if 'funct' in k or 'role' in k:
                reff_col = orig
                break
    if name_col is None:
        for k, orig in col_map.items():
            if 'na' in k or 'person' in k:
                name_col = orig
                break
    # build cleaned DataFrame
    if reff_col:
        cols = [reff_col] + ([name_col] if name_col else [])
        clean = candidate.loc[:, cols].copy()
        # normalize text: strip and convert non-strings to NA
        for c in clean.columns:
            clean[c] = clean[c].apply(lambda x: x.strip() if isinstance(x, str) and x.strip() != "" else pd.NA)
        # rename to exact desired column names
        rename_map = {reff_col: 'REFFUNCTIE'}
        if name_col:
            rename_map[name_col] = 'NAAM'
        clean = clean.rename(columns=rename_map)
        # ensure both columns exist
        if 'NAAM' not in clean.columns:
            clean['NAAM'] = pd.NA
    else:
        clean = pd.DataFrame(columns=['REFFUNCTIE', 'NAAM'])
    split_Officials = clean[['REFFUNCTIE', 'NAAM']].reset_index(drop=True)
    df_T = split_Officials.T.reset_index(drop=True)
    if df_T.shape[0] >= 2:
        df_promoted = pd.DataFrame([df_T.iloc[1].values], columns=df_T.iloc[0])
        return df_promoted
    else:
        print('Not enough rows in split_Officials.T to promote.')
        return pd.DataFrame()

In [ ]:
#FUNCTION GETTING EXTENDED TABLES BASES ON THE SEASON OR PART_SEASON#GETFULLTABLE(input_df, guid_col=None, wait_time=1)
def GETFULLTABLE(input_df, guid_col=None, wait_time=1):
    """
    Build per-match official mappings and attach them to a copy of input_df.
    Returns a new dataframe with columns:
      - officials_split_Officials : raw parsed officials (or None)
      - officials_map             : dict role -> name (or {})
      - OFF_<role>                : one column per discovered role
    Requires: enhanced_dynamic_scraper(url, wait_time) and parse_officials_from_html(html)
    """

    # validate input
    if input_df is None:
        raise ValueError("input_df must be provided")

    df_out = input_df.copy().reset_index(drop=True)

    # try to determine guid column if not supplied
    if guid_col is None:
        guid_col = None
        for c in df_out.columns:
            cl = c.lower()
            if 'wed' in cl and 'guid' in cl:
                guid_col = c
                break
        if guid_col is None:
            for c in df_out.columns:
                cl = c.lower()
                if 'guid' in cl or ('id' in cl and 'wed' in cl):
                    guid_col = c
                    break
        if guid_col is None:
            for c in df_out.columns:
                # look for long-string columns that may be GUIDs
                if df_out[c].astype(str).map(len).gt(20).any():
                    guid_col = c
                    break

    if guid_col is None:
        raise ValueError("Could not find a GUID column in input_df. Provide guid_col or inspect columns.")

    # helpers / containers
    officials_maps = []
    officials_raw = []
    all_roles = set()

    def make_match_url(wedguid):
        return f"https://vblweb.wisseq.eu/Home/MatchDetail?wedguid={wedguid}"

    # iterate and scrape
    for idx, row in df_out.iterrows():
        wedguid = row.get(guid_col)
        if pd.isna(wedguid) or str(wedguid).strip() == "":
            officials_maps.append({})
            officials_raw.append(None)
            continue

        wedguid_str = str(wedguid).strip()
        url = make_match_url(wedguid_str)

        try:
            html = enhanced_dynamic_scraper(url, wait_time=wait_time)
            if not html:
                officials_maps.append({})
                officials_raw.append(None)
                continue

            # parse officials from the page
            try:
                _, _, _, split_Officials = parse_officials_from_html(html)
            except Exception:
                officials_maps.append({})
                officials_raw.append(None)
                continue

            mapping = {}
            if isinstance(split_Officials, pd.DataFrame) and not split_Officials.empty:
                for _, r in split_Officials.iterrows():
                    role = r.get('REFFUNCTIE')
                    name = r.get('NAAM')
                    if pd.isna(role) or str(role).strip() == "":
                        continue
                    role_key = str(role).strip()
                    name_val = None if pd.isna(name) else str(name).strip()
                    mapping[role_key] = name_val
                    all_roles.add(role_key)

            officials_maps.append(mapping)
            officials_raw.append(split_Officials)

        except Exception:
            officials_maps.append({})
            officials_raw.append(None)

    # attach results to dataframe
    df_out['officials_split_Officials'] = officials_raw
    df_out['officials_map'] = officials_maps

    for role in sorted(all_roles):
        colname = f"OFF_{role}"
        df_out[colname] = [
            m.get(role, pd.NA) if isinstance(m, dict) else pd.NA for m in officials_maps
        ]

    return df_out

# Mathes

In [13]:
clubmatchesurl = "https://vblcb.wisseq.eu/VBLCB_WebService/data/OrgMatchesByGuid?issguid=BVBL1037"

In [14]:
# Create a DataFrame called `season` from available variables (payload / resp / Matches / Officials)
# Uses existing imports and variables in the notebook.
# Try fetching club matches via requests so the selection logic below can use `resp` / `payload`
fetch_url = None
if 'clubmatchesurl' in globals():
    fetch_url = clubmatchesurl
elif 'matchurl' in globals():
    fetch_url = matchurl

if fetch_url:
    print(f"📡 requests.get -> {fetch_url}")
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        resp = requests.get(fetch_url, headers=headers, timeout=30)
        print(f"📬 Response {resp.status_code} | Content-Type: {resp.headers.get('content-type')}")
        try:
            payload = resp.json()
            print(f"🔁 Parsed JSON into variable 'payload' (type: {type(payload).__name__})")
        except Exception:
            payload = None
            print("⚠️ Response not JSON — 'payload' set to None")
    except Exception as e:
        resp = None
        payload = None
        print(f"❌ requests.get failed: {e}")
else:
    resp = None
    payload = None
    print("⚠️ No URL found (clubmatchesurl or matchurl not defined)")


📡 requests.get -> https://vblcb.wisseq.eu/VBLCB_WebService/data/OrgMatchesByGuid?issguid=BVBL1037
📬 Response 200 | Content-Type: application/json; charset=utf-8
🔁 Parsed JSON into variable 'payload' (type: list)


In [15]:
# Transform payload (list of dicts) into a pandas DataFrame named `season`
if 'payload' in globals() and payload:
    season = pd.DataFrame(payload)
    print(f"✅ Created 'season' DataFrame with {len(season)} rows and {len(season.columns)} columns")
else:
    season = pd.DataFrame()
    print("⚠️ 'payload' not found or empty — created empty 'season' DataFrame")

    if not season.empty and 'jsDTCode' in season.columns:
        season = season.sort_values('jsDTCode', ascending=True).reset_index(drop=True)
        print(f"✅ 'season' sorted by 'jsDTCode' ascending")
    else:
        print("⚠️ Cannot sort: 'season' is empty or 'jsDTCode' missing")

season.head(10)  # Display first 50 rows for inspection
# Create DT2 by combining datumString and beginTijd, parse to datetime and sort
if season.empty:
    print("⚠️ 'season' is empty — nothing to sort")
else:
    # Normalize beginTijd: replace dots with colons, strip, fill empty with 00:00
    begin_fixed = (
        season['beginTijd']
        .fillna('')
        .astype(str)
        .str.strip()
        .replace('', '00:00')
        .str.replace('.', ':', regex=False)
    )
    
    # Combine date and time strings
    combined = season['datumString'].astype(str).str.strip() + ' ' + begin_fixed
    
    # Parse to datetime (dayfirst because datumString is dd-mm-yyyy)
    season['DT2'] = pd.to_datetime(combined, dayfirst=True, errors='coerce')
    
    # For any rows where time parsing failed, fallback to date only (00:00)
    mask_nat = season['DT2'].isna()
    if mask_nat.any():
        season.loc[mask_nat, 'DT2'] = pd.to_datetime(
            season.loc[mask_nat, 'datumString'], dayfirst=True, errors='coerce'
        )
    
    # Report parsing results
    nat_count = season['DT2'].isna().sum()
    print(f"✅ Created 'DT2' datetime column — {len(season) - nat_count} parsed, {nat_count} failed (NaT)")
    
    # Sort by DT2 and reset index
    season = season.sort_values('DT2', ascending=True).reset_index(drop=True)
    print("✅ 'season' sorted by 'DT2' (combined datumString + beginTijd)")

season.head(10)


now = pd.Timestamp.now().normalize()
end = now + pd.Timedelta(days=21)

mask_window = season['DT2'].notna() & (season['DT2'] >= now) & (season['DT2'] <= end)
season_nex2weeks = season.loc[mask_window].sort_values('DT2').reset_index(drop=True)




✅ Created 'season' DataFrame with 327 rows and 17 columns
✅ Created 'DT2' datetime column — 327 parsed, 0 failed (NaT)
✅ 'season' sorted by 'DT2' (combined datumString + beginTijd)


In [16]:
# Test the function with both spans
print("🎯 Testing unified export function...")

if 'payload' in globals() and payload:
    # Export all matches
    print("\n" + "="*50)
    print("📊 EXPORTING ALL MATCHES")
    print("="*50)
    all_file = export_matches(payload, "all")
    
    # Export next 2 weeks
    print("\n" + "="*50)
    print("📅 EXPORTING NEXT 2 WEEKS")
    print("="*50)
    weeks_file = export_matches(payload, "2W")
    
    print(f"\n🎉 Export completed!")
    if all_file:
        print(f"📄 All matches: {all_file}")
    if weeks_file:
        print(f"📄 Next 2 weeks: {weeks_file}")
else:
    print("❌ No payload data available for export")

🎯 Testing unified export function...

📊 EXPORTING ALL MATCHES
📊 Starting export with span: all
✅ SUCCESS! Exported 327 matches to 'matches_all.csv'
📊 Processed 327 total records

👀 Preview of exported matches:
  1. 07-08-2025 | 20.00 | BBC Haantjes Certifisc Oudenaarde HSE A | Blue Rocks Ronse-Kluisbergen HSE A | BBC Haantjes Oudenaarde OEFEN
  2. 09-08-2025 | 17.30 | G&V Breakpoint Basket Waregem DSE D | BBC Haantjes Certifisc Oudenaarde DSE A | Koninklijk Basket Team ION Waregem OEFEN
  3. 09-08-2025 | 18.30 | Oxaco BBC Boechout HSE B | BBC Haantjes Certifisc Oudenaarde HSE A | Beker van Vlaanderen Heren Poule G
  4. 10-08-2025 | 17.00 | BBC Haantjes Certifisc Oudenaarde HSE A | Midwest All-in Garden Tielt HSE A | BBC Haantjes Oudenaarde OEFEN
  5. 13-08-2025 | 20.30 | BBC Haantjes Certifisc Oudenaarde DSE A | DBC Osiris Okapi Aalst DSE B | BBC Haantjes Oudenaarde OEFEN
     ... and 322 more matches

📅 EXPORTING NEXT 2 WEEKS
📊 Starting export with span: 2W
📅 Date window: 2025-09-09 →

In [27]:
season_nex2weeks.head(3)
input = season_nex2weeks

In [28]:
data =input[0:16]

In [29]:
data.to_excel("season_nex2weeks.xlsx", index=False)
print("✅ Exported to 'season_nex2weeks.xlsx'")

✅ Exported to 'season_nex2weeks.xlsx'


In [18]:


def get_nextdays(season=globals().get('season'), days=6):
    """
    Return matches in `season` whose DT2 falls between now and now+days.
    season: pd.DataFrame (defaults to the global `season` if available)
    days: int number of days ahead to include
    """
    if season is None or season.empty:
        return pd.DataFrame()
    now = pd.Timestamp.now().normalize()
    end = now + pd.Timedelta(days=days)
    mask_window = season['DT2'].notna() & (season['DT2'] >= now) & (season['DT2'] <= end)
    return season.loc[mask_window].sort_values('DT2').reset_index(drop=True)


# run functions

In [28]:
input = get_nextdays(season=globals().get('season'), days=7)
fulldf = GETFULLTABLE(input)

🚀 Enhanced scraper for dynamic content...
❌ Enhanced scraper failed: [WinError 4551] Your organization used Device Guard to block this app. Contact your support person for more info
🚀 Enhanced scraper for dynamic content...
❌ Enhanced scraper failed: [WinError 4551] Your organization used Device Guard to block this app. Contact your support person for more info
🚀 Enhanced scraper for dynamic content...
❌ Enhanced scraper failed: [WinError 4551] Your organization used Device Guard to block this app. Contact your support person for more info
🚀 Enhanced scraper for dynamic content...
❌ Enhanced scraper failed: [WinError 4551] Your organization used Device Guard to block this app. Contact your support person for more info
🚀 Enhanced scraper for dynamic content...
❌ Enhanced scraper failed: [WinError 4551] Your organization used Device Guard to block this app. Contact your support person for more info
🚀 Enhanced scraper for dynamic content...
❌ Enhanced scraper failed: [WinError 4551] Your 

In [30]:
import ast

# adapt fulldf in-place (safe copy)
fulldf = fulldf.copy()

# drop unwanted columns if present
_to_drop = ['tTGUID', 'tUGUID', 'tTKleur', 'tUKleur', 'officials_split_Officials', 'officials_map']
present_drop = [c for c in _to_drop if c in fulldf.columns]
if present_drop:
    fulldf.drop(columns=present_drop, inplace=True)

# rename columns if present
rename_map = {}
if 'tTNaam' in fulldf.columns:
    rename_map['tTNaam'] = 'Thuisploeg'
if 'tUNaam' in fulldf.columns:
    rename_map['tUNaam'] = 'uitploeg'
if 'DT2' in fulldf.columns:
    rename_map['DT2'] = 'tijdstip'

if rename_map:
    fulldf.rename(columns=rename_map, inplace=True)

# quick check
# fulldf.info()
# fulldf
def _extract_off(x, idx):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return pd.NA
    if isinstance(x, (list, tuple)):
        try:
            v = x[idx]
            return v.strip() if isinstance(v, str) and v.strip() != "" else (pd.NA if v == "" else v)
        except Exception:
            return pd.NA
    if isinstance(x, str):
        s = x.strip()
        # try to parse stringified list
        try:
            val = ast.literal_eval(s)
            if isinstance(val, (list, tuple)):
                return _extract_off(val, idx)
        except Exception:
            s2 = s.strip("[]()")
            parts = [p.strip().strip("'\"") for p in s2.split(",") if p.strip() != ""]
            try:
                return parts[idx]
            except Exception:
                return pd.NA
    return pd.NA

if 'wedOff' in fulldf.columns:
    fulldf['wedOFF1'] = fulldf['wedOff'].apply(lambda x: _extract_off(x, 0))
    fulldf['wedOFF2'] = fulldf['wedOff'].apply(lambda x: _extract_off(x, 1))
else:
    print("⚠️ Column 'wedOff' not found in fulldf")

# fulldf[['wedOff', 'wedOFF1', 'wedOFF2']].head(10)
fulldf

,guid,wedID,Thuisploeg,uitploeg,datumString,jsDTCode,accGUID,accNaam,pouleGUID,pouleNaam,uitslag,beginTijd,wedOff,tijdstip,wedOFF1,wedOFF2
0,BVBL25261037OR00040520,OR0004052001,BBC Haantjes Certifisc Oudenaarde G12 C,ABO LDP Donza G12 B,06-09-2025,1757152800000,BVBL100343,Rode Los,BVBL25261037OR000405,BBC Haantjes Oudenaarde OEFEN,,10.00,None,2025-09-06 10:00:00,<NA>,<NA>
1,BVBL25269130BOVG121609,BOVG12160901,BBC Haantjes Certifisc Oudenaarde G12 A,BC Black Boys Erpe-Mere G12 A,06-09-2025,1757161800000,BVBL100343,Rode Los,BVBL25269130BOVG1216,Beker van Oost-Vlaanderen U12 Gemengd 1/16 Finale,,12.30,None,2025-09-06 12:30:00,<NA>,<NA>
2,BVBL25269100BLAG14PCAC,BLAG14PCAC03,BBC Haantjes Certifisc Oudenaarde G14 A,Bavi Vilvoorde G14 A,06-09-2025,1757165400000,BVBL100343,Rode Los,BVBL25269100BLAG14PC,Beker van Vlaanderen U14 Poule C,,13.30,"[Pascal Robbe, Luc Defour]",2025-09-06 13:30:00,Pascal Robbe,Luc Defour
3,BVBL25269100BLAJ16PBAC,BLAJ16PBAC03,BBC Haantjes Certifisc Oudenaarde J16 A,Basket Malle J16 A,06-09-2025,1757169900000,BVBL100343,Rode Los,BVBL25269100BLAJ16PB,Beker van Vlaanderen U16 Poule B,,14.45,"[Marc Temmerman, Noel Scheire]",2025-09-06 14:45:00,Marc Temmerman,Noel Scheire
4,BVBL25269130BOVJ161601,BOVJ16160101,KBBC Bavi Gent J16 B,BBC Haantjes Certifisc Oudenaarde J16 B,06-09-2025,1757172600000,BVBL100305,Sporthal Neptunus,BVBL25269130BOVJ1616,Beker van Oost-Vlaanderen U16 Jongens 1/16 Finale,,15.30,"[Johan Delboo, Bor Cobbaut]",2025-09-06 15:30:00,Johan Delboo,Bor Cobbaut
5,BVBL25269100BLAJ18PNAC,BLAJ18PNAC03,BBC Haantjes Certifisc Oudenaarde J18 A,Phantoms Basket Boom J18 A,06-09-2025,1757178000000,BVBL100343,Rode Los,BVBL25269100BLAJ18PN,Beker van Vlaanderen U18 Poule N,,17.00,"[Kris Dierick, Geert Citters]",2025-09-06 17:00:00,Kris Dierick,Geert Citters
6,BVBL25261037OR00040505,OR0004050501,BBC Haantjes Certifisc Oudenaarde HSE B,Blue Rocks Ronse-Kluisbergen HSE B,06-09-2025,1757187000000,BVBL100343,Rode Los,BVBL25261037OR000405,BBC Haantjes Oudenaarde OEFEN,,19.30,"[Kris Dierick, Geert Citters]",2025-09-06 19:30:00,Kris Dierick,Geert Citters
7,BVBL25269100BLAHSEPGBD,BLAHSEPGBD04,Geranimo Bornem Basket HSE A,BBC Haantjes Certifisc Oudenaarde HSE A,06-09-2025,1757189700000,BVBL100381,Sporthal Breeven,BVBL25269100BLAHSEPG,Beker van Vlaanderen Heren Poule G,,20.15,"[Vincent Villé, Jonathan De Rese]",2025-09-06 20:15:00,Vincent Villé,Jonathan De Rese
8,BVBL25261257OR00189602,OR0018960201,BC Grimbergen DSE A,BBC Haantjes Certifisc Oudenaarde DSE A,06-09-2025,1757190600000,BVBL100200,Sporthal Borgt,BVBL25261257OR001896,BC Grimbergen OEFEN,,20.30,"[Jarne Buelens, Dylan Marcon]",2025-09-06 20:30:00,Jarne Buelens,Dylan Marcon
9,BVBL25269130BOVG141609,BOVG14160901,Basket Meetjesland G14 A,BBC Haantjes Certifisc Oudenaarde G14 B,07-09-2025,1757244600000,BVBL100331,Sporthal Kaprijke Berkakker,BVBL25269130BOVG1416,Beker van Oost-Vlaanderen U14 Gemengd 1/16 Fin...,,11.30,"[Robert Van Eenaeme, Vic Regelbrugge]",2025-09-07 11:30:00,Robert Van Eenaeme,Vic Regelbrugge


In [31]:
# remove duplicate rows in fulldf based on 'guid' (keep first occurrence)
if 'fulldf' not in globals():
    raise NameError("fulldf not found in globals()")

if 'guid' not in fulldf.columns:
    raise KeyError("Column 'guid' not found in fulldf")

_before = len(fulldf)
fulldf.drop_duplicates(subset=['guid'], keep='first', inplace=True)
fulldf.reset_index(drop=True, inplace=True)
_after = len(fulldf)

print(f"✅ Removed {_before - _after} duplicate rows based on 'guid'. Remaining rows: {_after}")

✅ Removed 0 duplicate rows based on 'guid'. Remaining rows: 16


In [32]:
ts = pd.Timestamp.now().strftime('%d%m%y_%H%M')
filename = f"bball_refs_{ts}.xlsx"
fulldf.to_excel(filename, index=False)
print(f"✅ Saved: {filename}")

✅ Saved: bball_refs_020925_1537.xlsx
